# Train Brainrot Translator FLAN-T5

Use this notebook in Google Colab with GPU enabled. Upload `training_dataset_final_local_only.csv`, train `google/flan-t5-small`, then download `brainrot-translator-v1.zip`. Extract it into `D:\brainrot_translator\models\brainrot-translator-v1`.

In [ ]:
!pip -q install -U transformers datasets accelerate sentencepiece evaluate safetensors

In [ ]:
import transformers
print("transformers:", transformers.__version__)

In [ ]:
from google.colab import files

uploaded = files.upload()
DATASET_PATH = next(iter(uploaded.keys()))
print("dataset:", DATASET_PATH)

In [ ]:
import pandas as pd

REQUIRED_COLUMNS = ["input_text", "target_text", "task_type", "source", "quality_label", "reason"]

df = pd.read_csv(DATASET_PATH)
missing = [column for column in REQUIRED_COLUMNS if column not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.dropna(subset=["input_text", "target_text"])
df["input_text"] = df["input_text"].astype(str).str.strip()
df["target_text"] = df["target_text"].astype(str).str.strip()
df = df[df["input_text"].ne("") & df["target_text"].ne("")]

bad_prefix = ~df["input_text"].str.startswith("Convert brainrot English to normal English:")
if bad_prefix.any():
    raise ValueError(f"Rows with invalid prompt prefix: {int(bad_prefix.sum())}")
if len(df) < 1000:
    raise ValueError(f"Dataset is too small for this training run: {len(df)} rows")

print("rows, columns:", df.shape)
print("task_type counts:")
print(df["task_type"].value_counts())
display(df.sample(min(20, len(df)), random_state=42))

In [ ]:
import torch
from datasets import Dataset
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq, get_linear_schedule_with_warmup

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU in Colab: Runtime > Change runtime type > T4 GPU.")

print("gpu:", torch.cuda.get_device_name(0))

MODEL_NAME = "google/flan-t5-small"
OUTPUT_DIR = "brainrot-translator-v1"
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

MANUAL_GOLD_OVERSAMPLE = 6
manual_gold = df[df["source"].eq("manual_gold_examples")]
if manual_gold.empty:
    print("manual_gold_examples rows: 0; training without oversampling")
    train_source_df = df
else:
    train_source_df = pd.concat([df, *([manual_gold] * (MANUAL_GOLD_OVERSAMPLE - 1))], ignore_index=True)
    print("manual_gold_examples rows:", len(manual_gold))
    print("training rows after manual gold oversampling:", len(train_source_df))

dataset = Dataset.from_pandas(train_source_df[["input_text", "target_text"]], preserve_index=False).train_test_split(test_size=0.15, seed=42)

def preprocess(batch):
    inputs = tokenizer(batch["input_text"], max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(text_target=batch["target_text"], max_length=MAX_TARGET_LENGTH, truncation=True)
    inputs["labels"] = labels["input_ids"]
    return inputs

tokenized = dataset.map(preprocess, batched=True, remove_columns=dataset["train"].column_names)
train_label_lengths = [len(labels) for labels in tokenized["train"]["labels"]]
eval_label_lengths = [len(labels) for labels in tokenized["test"]["labels"]]
print("train rows:", len(tokenized["train"]), "eval rows:", len(tokenized["test"]))
print("target token length min/max:", min(train_label_lengths), max(train_label_lengths))
if min(train_label_lengths) == 0 or min(eval_label_lengths) == 0:
    raise ValueError("At least one target_text produced zero target tokens.")

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [ ]:
!rm -rf training-runs brainrot-translator-v1 brainrot-translator-v1.zip

In [ ]:
BATCH_SIZE = 8
EPOCHS = 4
LEARNING_RATE = 5e-5

device = torch.device("cuda")
model.to(device)

train_loader = DataLoader(tokenized["train"], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
eval_loader = DataLoader(tokenized["test"], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, total_steps // 20),
    num_training_steps=total_steps,
)

def move_to_device(batch):
    return {key: value.to(device) for key, value in batch.items()}

def evaluate():
    model.eval()
    losses = []
    with torch.no_grad():
        for batch in eval_loader:
            batch = move_to_device(batch)
            outputs = model(**batch)
            loss = outputs.loss.detach().float()
            if torch.isfinite(loss):
                losses.append(loss.item())
    model.train()
    if not losses:
        raise RuntimeError("Evaluation produced no finite losses. Check target_text values and tokenization.")
    return sum(losses) / len(losses)

history = []
model.train()
for epoch in range(1, EPOCHS + 1):
    running_loss = 0.0
    finite_steps = 0
    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for step, batch in enumerate(progress, start=1):
        batch = move_to_device(batch)
        outputs = model(**batch)
        loss = outputs.loss
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite training loss at epoch {epoch}, step {step}: {loss.item()}")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        running_loss += loss.detach().float().item()
        finite_steps += 1
        if step % 25 == 0:
            progress.set_postfix(train_loss=running_loss / finite_steps)

    train_loss = running_loss / max(1, finite_steps)
    eval_loss = evaluate()
    history.append({"epoch": epoch, "train_loss": train_loss, "eval_loss": eval_loss})
    print(f"epoch={epoch} train_loss={train_loss:.4f} eval_loss={eval_loss:.4f}")

history

In [ ]:
def translate(text):
    prompt = f"Convert brainrot English to normal English: {text.strip()}"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=128).to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=128, num_beams=4, do_sample=False)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

samples = [
    "skill issue",
    "he has rizz",
    "that movie was mid",
    "bro got cooked in the replies",
    "she ate and left no crumbs",
]

for sample in samples:
    print(sample, "=>", translate(sample))

In [ ]:
model.save_pretrained(OUTPUT_DIR, safe_serialization=True)
tokenizer.save_pretrained(OUTPUT_DIR)
!zip -qr brainrot-translator-v1.zip brainrot-translator-v1
files.download("brainrot-translator-v1.zip")